# NER Baseline using mBERT

Trying multilingual BERT for NER task.

## Install stuff

In [ ]:
# Uncomment and run if you haven't installed these yet
# !pip install transformers datasets seqeval torch accelerate -q


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# imports
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None - using CPU")

CUDA available: True
GPU: NVIDIA GeForce RTX 3050


## Imports needed

In [ ]:
# loading libraries and helper functions
import os
import numpy as np
from pathlib import Path

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from datasets import Dataset
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

e:\pyth\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All imports successful!


## Config

In [ ]:
# paths for datasets and outputs
# ── File paths 
DATA_DIR   = Path('E:/work/NLP')   # adjust if needed
TRAIN_FILE = DATA_DIR / 'en_ewt-ud-train.iob2'
DEV_FILE   = DATA_DIR / 'en_ewt-ud-dev.iob2'
TEST_FILE  = DATA_DIR / 'en_ewt-ud-test-masked.iob2'

# ── Model 
MODEL_NAME = 'bert-base-multilingual-cased'
OUTPUT_DIR = './ner_model_output'

# ── NER labels 
# O     = not an entity
# B-XXX = Beginning of entity type XXX
# I-XXX = Inside (continuation) of entity type XXX
LABEL_LIST = ['O', 'B-PER', 'I-PER', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG']
LABEL_TO_ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

print('Labels:', LABEL_LIST)
print('Label → ID mapping:', LABEL_TO_ID)

Labels: ['O', 'B-PER', 'I-PER', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG']
Label → ID mapping: {'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-LOC': 3, 'I-LOC': 4, 'B-ORG': 5, 'I-ORG': 6}


## Reading the data files

Dataset is in conll/iob style format so reading line by line.

In [ ]:
# function for reading the conll/iob2 style files
def read_iob2_file(filepath, is_test=False):
    """
    Reads an IOB2 file and returns a list of sentences.
    Each sentence: {'tokens': ['Where', 'is', ...], 'labels': ['O', 'O', ...]}
    """
    sentences = []
    current_tokens, current_labels = [], []

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')

            if line.startswith('#'):   # Skip comment lines
                continue

            if line.strip() == '':    # Blank line = end of sentence
                if current_tokens:
                    sentences.append({'tokens': current_tokens, 'labels': current_labels})
                    current_tokens, current_labels = [], []
                continue

            parts = line.split('\t')
            if len(parts) < 3:
                continue

            word  = parts[1]
            label = parts[2]

            # Test file has masked labels; treat as 'O'
            if label == '-' or label not in LABEL_LIST:
                label = 'O'

            current_tokens.append(word)
            current_labels.append(label)

    if current_tokens:  # Don't forget the last sentence
        sentences.append({'tokens': current_tokens, 'labels': current_labels})

    return sentences


train_data = read_iob2_file(TRAIN_FILE)
dev_data   = read_iob2_file(DEV_FILE)
test_data  = read_iob2_file(TEST_FILE, is_test=True)

print(f'Training sentences : {len(train_data)}')
print(f'Dev sentences      : {len(dev_data)}')
print(f'Test sentences     : {len(test_data)}')
print('Example sentence:')
print('  Tokens:', train_data[0]['tokens'])
print('  Labels:', train_data[0]['labels'])

Training sentences : 12543
Dev sentences      : 2001
Test sentences     : 2077

Example sentence:
  Tokens: ['Where', 'in', 'the', 'world', 'is', 'Iguazu', '?']
  Labels: ['O', 'O', 'O', 'O', 'O', 'B-LOC', 'O']


## Load tokenizer

In [ ]:
# loading tokenizer from huggingface
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Vocabulary size: {tokenizer.vocab_size:,}')

# Demonstrate the subword problem
example = ['Iguazu', 'Falls', 'is', 'beautiful']
encoded = tokenizer(example, is_split_into_words=True)
tokens  = tokenizer.convert_ids_to_tokens(encoded['input_ids'])
print()
print('Demo — subword tokenisation:')
print('  Input words:', example)
print('  Subword tokens:', tokens)
print('  word_ids():', encoded.word_ids())
print('  (None = special token [CLS]/[SEP], number = which word it came from)')

Loading tokenizer...
Vocabulary size: 119,547

Demo — subword tokenisation:
  Input words: ['Iguazu', 'Falls', 'is', 'beautiful']
  Subword tokens: ['[CLS]', 'I', '##gua', '##zu', 'Falls', 'is', 'beautiful', '[SEP]']
  word_ids(): [None, 0, 0, 0, 1, 2, 3, None]
  (None = special token [CLS]/[SEP], number = which word it came from)


## Tokenization + label alignment

This was kinda confusing because tokenizer breaks words into smaller tokens.

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples['tokens'],
        truncation=True,
        max_length=512,
        is_split_into_words=True,
        padding=False,
    )
#the subword tag is -100 so the model ignore it
    all_labels = []
    for i, label_list in enumerate(examples['labels']):
        word_ids = tokenized.word_ids(batch_index=i)
        aligned, prev_word_idx = [], None

        for word_idx in word_ids:
            if word_idx is None:      # special token
                aligned.append(-100)
            elif word_idx != prev_word_idx:             # first subword of word
                aligned.append(LABEL_TO_ID[label_list[word_idx]])
            else:# continuation subword
                aligned.append(-100)
            prev_word_idx = word_idx

        all_labels.append(aligned)

    tokenized['labels'] = all_labels
    return tokenized


#Build HuggingFace Datasets and apply the token thingy
def make_hf_dataset(data_list):
    return Dataset.from_dict({
        'tokens': [d['tokens'] for d in data_list],
        'labels': [d['labels'] for d in data_list],
    })

print('Tokenising datasets (this may take a minute)...')
tokenized_train = make_hf_dataset(train_data).map(
    tokenize_and_align_labels, batched=True, remove_columns=['tokens', 'labels'])
tokenized_dev = make_hf_dataset(dev_data).map(
    tokenize_and_align_labels, batched=True, remove_columns=['tokens', 'labels'])
tokenized_test = make_hf_dataset(test_data).map(
    tokenize_and_align_labels, batched=True, remove_columns=['tokens', 'labels'])

print(f'Train tokenised examples : {len(tokenized_train)}')
print(f'Dev   tokenised examples : {len(tokenized_dev)}')
print(f'Test  tokenised examples : {len(tokenized_test)}')
# took me a bit to understand this part honestly


Tokenising datasets (this may take a minute)...


Map: 100%|██████████| 2077/2077 [00:00<00:00, 30630.10 examples/s]

Train tokenised examples : 12543
Dev   tokenised examples : 2001
Test  tokenised examples : 2077


## Load model

In [ ]:
import torch  
#← add this line

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
    ignore_mismatched_sizes=True,
)

model = model.to(device)
print(f"Model is on: {next(model.parameters()).device}")



Using device: cuda


e:\pyth\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\namho\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 8944.82it/s]
BertForTokenClassification LOAD REPORT from: bert-base-multiling

Model is on: cuda:0


## Metrics

In [ ]:
#Now we are gonna compute precision/recall/F1
def compute_metrics(eval_preds):
    """
    Called by the Trainer after each epoch.
    Converts raw logits → label strings → seqeval F1.
    """
    logits, true_labels = eval_preds
    predictions = np.argmax(logits, axis=-1)   # pick the highest-score label

    true_strs, pred_strs = [], []
    for pred_seq, true_seq in zip(predictions, true_labels):
        true_s, pred_s = [], []
        for p, t in zip(pred_seq, true_seq):
            if t == -100:   # skip special tokens / subword continuations
                continue
            true_s.append(ID_TO_LABEL[t])
            pred_s.append(ID_TO_LABEL[p])
        true_strs.append(true_s)
        pred_strs.append(pred_s)

    return {
        'precision': precision_score(true_strs, pred_strs),
        'recall':    recall_score(true_strs, pred_strs),
        'f1':        f1_score(true_strs, pred_strs),
    }

✅ compute_metrics defined


## Training setup

Using Trainer API because it is easier than writing full training loop manually.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # force GPU 0

import torch
print("GPU being used:", torch.cuda.get_device_name(0))

GPU being used: NVIDIA GeForce RTX 3050


In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    use_cpu=False,
    fp16=True,  #mixed precision (huge speedup on NVIDIA!)
    
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    # evaluate after every epoch
    eval_strategy='epoch',        
    save_strategy='epoch',
    load_best_model_at_end=True,                    #keep the checkpoint with best F1
    metric_for_best_model='f1',

    weight_decay=0.01,
    warmup_steps=500,
    logging_steps=50,
    report_to='none', #disable wandb or tensorboard
    seed=42,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    processing_class=tokenizer,  
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)




✅ Trainer ready


## Train model

In [ ]:
import torch
print(torch.cuda.is_available())        
print(torch.cuda.get_device_name(0)) 

# FirstOne should print 'True'
 # SecondOne should print your 'GPU' name


True
NVIDIA GeForce RTX 3050


In [ ]:
#Starts Training
print('Starting training...')
print('(On CPU this takes ~30-60 min. Grab a coffee! ☕)')
print()

train_result = trainer.train()

print()
print('Training complete!')
print(f'Final training loss: {train_result.training_loss:.4f}')

Starting training...
(On CPU this takes ~30-60 min. Grab a coffee! ☕)



Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.040622,0.066687,0.753359,0.812629,0.781873
2,0.026340,0.066027,0.794243,0.771222,0.782563
3,0.011138,0.067346,0.808777,0.801242,0.804992


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.50s/it]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer


Training complete!
Final training loss: 0.0421


## Eval on dev set

Wanted to check if model is overfitting or not.

In [ ]:


# Evaluating on validation set manually
#Re-run predictions directly without calling trainer.evaluate()

print("Evaluating on dev set...")
dev_preds_raw = trainer.predict(tokenized_dev)
dev_preds     = np.argmax(dev_preds_raw.predictions, axis=-1)
dev_true      = dev_preds_raw.label_ids

true_strs, pred_strs = [], []
for pred_seq, true_seq in zip(dev_preds, dev_true):
    t, p = [], []
    for pred_id, true_id in zip(pred_seq, true_seq):
        if true_id == -100:
            continue
        t.append(ID_TO_LABEL[true_id])
        p.append(ID_TO_LABEL[pred_id])
    true_strs.append(t)
    pred_strs.append(p)

print(f"  F1 : {f1_score(true_strs, pred_strs):.4f}")
print(f"  Precision : {precision_score(true_strs, pred_strs):.4f}")
print(f"  Recall    : {recall_score(true_strs, pred_strs):.4f}")
print()
print("Per-entity-type breakdown:")
print(classification_report(true_strs, pred_strs))

Evaluating on dev set...
  F1 : 0.8050
  Precision : 0.8088
  Recall    : 0.8012

Per-entity-type breakdown:
              precision    recall  f1-score   support

         LOC       0.82      0.81      0.81       399
         ORG       0.67      0.64      0.66       224
         PER       0.88      0.90      0.89       343

   micro avg       0.81      0.80      0.80       966
   macro avg       0.79      0.78      0.79       966
weighted avg       0.81      0.80      0.80       966



## Make predictions

In [ ]:
print('Generating test predictions...')
test_preds_raw = trainer.predict(tokenized_test)
test_preds     = np.argmax(test_preds_raw.predictions, axis=-1)

#Re-tokenise to get word_ids (needed to align preds back to words)
test_tokenized_for_ids = tokenizer(
    [d['tokens'] for d in test_data],
    truncation=True, max_length=512,
    is_split_into_words=True, padding=False,
)

output_predictions = []
for i, pred_seq in enumerate(test_preds):
    word_ids = test_tokenized_for_ids.word_ids(batch_index=i)
    sentence_preds, prev_word_idx = [], None
    for j, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        if word_idx != prev_word_idx:#first subword = our prediction
            sentence_preds.append(ID_TO_LABEL[pred_seq[j]])
        prev_word_idx = word_idx
    output_predictions.append(sentence_preds)

#Saving
output_path = './test_predictions.txt'
with open(output_path, 'w', encoding='utf-8') as f:
    for sentence, preds in zip(test_data, output_predictions):
        for token, pred in zip(sentence['tokens'], preds):
            f.write(f'{token}\t{pred}\n')
        f.write('\n')

print(f'Predictions saved to: {output_path}')
print('Preview (first 10 lines):')
with open(output_path) as f:
    for i, line in enumerate(f):
        print(' ', line, end='')
        if i > 10: break

Generating test predictions...


e:\pyth\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Predictions saved to: ./test_predictions.txt
Preview (first 10 lines):
  What	O
  is	O
  this	O
  Miramar	O
  ?	O
  
  It	O
  is	O
  a	O
  place	O
  in	O
  Argentina	B-LOC


## Save everything

In [ ]:
#Saving Trained model Locaally

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model and tokenizer saved to: {OUTPUT_DIR}/')

#To reload later:
#   model= AutoModelForTokenClassification.from_pretrained(OUTPUT_DIR)
#tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.37s/it]

Model and tokenizer saved to: ./ner_model_output/


## Other languages

In [ ]:
#Quick example for checking another language afterwards
# Uploading second language's data with one line 
#from datasets import load_dataset
#
#######################################################
#
#Spanish = load_dataset('conll2002', 'es') # Spanish
# german = load_dataset('conll2003') # German
# french    = load_dataset('wikiann', 'fr') # French
# Chinese= load_dataset('wikiann', 'zh')      #Chinese
#
########################################################
#
#Note: WikiANN uses different label names ('PER', 'LOC', 'ORG' without B-/I- prefix)
# We might need a small label-remapping step.

# Zero-shot evaluation on another language 
# Once we have a tokenised dataset for the new language, just run:
# results = trainer.evaluate(eval_dataset=tokenized_spanish_test)
# print(results)
###################
# That's all! Same model, different language.

print('See the comments above for how to extend to other languages.')
print('Fill in your results table:')
print(f'{"Language":<12} {"F1 (zero-shot)":<18} {"F1 (fine-tuned)"}')
print('-' * 46)
for lang in ['English', 'Spanish', 'German', 'French / Chinese']:
    print(f'{lang:<12} {"???":<18} {"???"}')